In [1]:
import pandas as pd
pd.set_option('max_columns', None)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import scipy.stats as stats
sns.set()

%matplotlib inline 


# import sklearn
from sklearn import tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
confusion_matrix, f1_score, roc_auc_score, make_scorer)
from sklearn.model_selection import GridSearchCV 
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [2]:
df = pd.read_csv('data/allegations_202007271729.csv')
df.head().T

,0,1,2,3,4
unique_mos_id,10004,10007,10007,10007,10009
first_name,Jonathan,John,John,John,Noemi
last_name,Ruiz,Sears,Sears,Sears,Sierra
command_now,078 PCT,078 PCT,078 PCT,078 PCT,078 PCT
shield_no,8409,5952,5952,5952,24058
complaint_id,42835,24601,24601,26146,40253
month_received,7,11,11,7,8
year_received,2019,2011,2011,2012,2018
month_closed,5,8,8,9,2
year_closed,2020,2012,2012,2013,2019


In [3]:
# Remove duplicate rows 
print(f'Shape before removing duplicates: {df.shape}')
df.drop_duplicates(inplace=True)
# Sanity check 
print(f'Num duplicates remaining: {df.duplicated().sum()}')
print(f'Shape after removing duplicates: {df.shape}')

Shape before removing duplicates: (33358, 27)
Num duplicates remaining: 0
Shape after removing duplicates: (32727, 27)


In [4]:
# create column for unsub/exonerated 
condition = (df['board_disposition']=='Unsubstantiated') | (df['board_disposition']=='Exonerated')
df['unsubst_or_exonerated'] = np.where(condition, 1, 0)

In [5]:
#fillin na
df['precinct'] = df['precinct'].fillna(-999)

## Split our data into training and testing sets. 
The data will be split based on date. 

In [6]:
# Create a dataframe with training data
df_train = df[df['year_received'] < 2017]
df_test = df[df['year_received'] > 2016] 
print(f'Earliest year in training dataframe: {df_train["year_received"].min()} \nLatest year in training dataframe: {df_train["year_received"].max()}')
print(f'Earliest year in testing dataframe: {df_test["year_received"].min()} \nLatest year in testing dataframe: {df_test["year_received"].max()}')
print(f'Training dataframe shape: {df_train.shape} \nTesting dataframe shape: {df_test.shape}')
# make sure dataframe samples add up to the total of the original dataframe
print(df_train.shape[0]+df_test.shape[0]==df.shape[0])

Earliest year in training dataframe: 1985 
Latest year in training dataframe: 2016
Earliest year in testing dataframe: 2017 
Latest year in testing dataframe: 2020
Training dataframe shape: (26827, 28) 
Testing dataframe shape: (5900, 28)
True


In [7]:
def prep_data(df):
    """
    Transforms data to suit modeling needs. Returns df. 
    """
    # Fill null values in numeric rows with median to avoid impact of outliers 
    for label, content in df.items():
        if pd.api.types.is_numeric_dtype(content):
            if pd.isnull(content).sum():
                # Add a column to indicate if data was missing
                df[label+"_is_missing"] = pd.isnull(content)
                # Fill missing numeric values with median
                df[label] = content.fillna(content.median())
    
        # Turn object/string rows into categories 
        if not pd.api.types.is_numeric_dtype(content):
            # Column to indicate if data was missing
            df[label+"_is_missing"] = pd.isnull(content)
            # add +1 to the category code because pandas encodes missing vals with -1
            # This way NaN values will be codes as 1 
            df[label] = pd.Categorical(content).codes+1
    
    return df
# Use prep_data function to categories and fill NaN vals in training df
df_train = prep_data(df_train)
# Use prep_data function to categories and fill NaN vals in testing df
df_test = prep_data(df_test)

/Users/xiaqianzhang/opt/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/Users/xiaqianzhang/opt/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/Users/xiaqianzhang/opt/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the cav

In [8]:
# take the model with the input that have the highest importance

df_train = df_train[['year_received','complaint_id', 'allegation', 'complainant_age_incident', 
                    'precinct', 'unique_mos_id', 'board_disposition','unsubst_or_exonerated']].copy()
df_test = df_test[['year_received','complaint_id', 'allegation', 'complainant_age_incident', 
                    'precinct', 'unique_mos_id', 'board_disposition','unsubst_or_exonerated']].copy()

df_train.head().T

,1,2,3,6,12
year_received,2011.0,2011.0,2012.0,2015.0,2016.0
complaint_id,24601.0,24601.0,26146.0,33969.0,35092.0
allegation,1.0,63.0,61.0,79.0,75.0
complainant_age_incident,26.0,26.0,45.0,34.0,30.0
precinct,67.0,67.0,67.0,78.0,79.0
unique_mos_id,10007.0,10007.0,10007.0,10014.0,10026.0
board_disposition,2.0,2.0,2.0,7.0,11.0
unsubst_or_exonerated,0.0,0.0,0.0,0.0,1.0


In [10]:
#set up the train and test value
X_train, y_train = df_train.drop(['year_received','board_disposition','unsubst_or_exonerated'], axis=1), df_train['unsubst_or_exonerated']
X_test, y_test = df_test.drop(['year_received','board_disposition','unsubst_or_exonerated'], axis=1), df_test['unsubst_or_exonerated']

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((26827, 5), (26827,), (5900, 5), (5900,))

In [11]:
def evaluate_class_model(model): 
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_true=y_test, y_pred=y_pred, )
    print("Accuracy Score: %f" % accuracy)

    precision = precision_score(y_true=y_test, y_pred=y_pred, average='micro')
    print("Precision Score: %f" % precision)

    recall = recall_score(y_true=y_test, y_pred=y_pred, average='micro')
    print("Recall Score: %f" % recall)

    f1 = f1_score(y_true=y_test, y_pred=y_pred, average='micro')
    print('F1 Score: %f' % f1)

#     Calculate predicted probabilities, keep only probability for when class = 1
    y_pred_proba = model.predict_proba(X_test)[:,1]
    auc = roc_auc_score(y_true=y_test, y_score=y_pred_proba, multi_class='ovo')
    print(f'AUC Score: %f' % auc)

In [12]:
#with random forest classifier
model = RandomForestClassifier(n_estimators=20, criterion='gini', 
                              min_samples_split=2, max_depth=100)
model.fit(X_train, y_train)
evaluate_class_model(model)

Accuracy Score: 0.602881
Precision Score: 0.602881
Recall Score: 0.602881
F1 Score: 0.602881
AUC Score: 0.545280


In [16]:
# increase the accuracy score with best params
# {'criterion': 'gini', 'max_depth': 2, 'max_features': 2, 'min_samples_split': 10, 'n_estimators': 10}
model = RandomForestClassifier(n_estimators=10, criterion='gini', 
                              max_features=2, max_depth=2,
                              min_samples_split=10)
model.fit(X_train, y_train)
evaluate_class_model(model)

Accuracy Score: 0.718644
Precision Score: 0.718644
Recall Score: 0.718644
F1 Score: 0.718644
AUC Score: 0.541058


In [17]:
# check feature importance 
selected_features = list(df_train.drop(['year_received','board_disposition','unsubst_or_exonerated'], axis=1))
feature_imp = pd.DataFrame.from_dict( {'feature_importance': model.feature_importances_,
                                       'feature':selected_features }).sort_values('feature_importance', ascending=False)
feature_imp

,feature_importance,feature
0,0.471973,complaint_id
1,0.442720,allegation
3,0.048821,precinct
2,0.019503,complainant_age_incident
4,0.016983,unique_mos_id


## Test out with prediction function

In [18]:
condition1 = df_train.unsubst_or_exonerated == 0
df_train[condition1].head()

,year_received,complaint_id,allegation,complainant_age_incident,precinct,unique_mos_id,board_disposition,unsubst_or_exonerated
1,2011,24601,1,26.0,67.0,10007,2,0
2,2011,24601,63,26.0,67.0,10007,2,0
3,2012,26146,61,45.0,67.0,10007,2,0
6,2015,33969,79,34.0,78.0,10014,7,0
20,2015,32450,16,29.0,79.0,10026,3,0


In [19]:
df_train.dtypes

year_received                 int64
complaint_id                  int64
allegation                     int8
complainant_age_incident    float64
precinct                    float64
unique_mos_id                 int64
board_disposition              int8
unsubst_or_exonerated         int64
dtype: object

In [38]:
pred = model[6].predict(X_test)
print(pred[:300])
print()
print(y_test.head(10))

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

0     0
4     0
5     0
7     1
8     1
9     0
10    0
11    1
30    1
31    1
Name:

In [21]:
# as follow complaint_id, allegation, complainant_age_incident, precinct, and unique_mos_id
my_prediction = [[24601,1, 26.0, 67.0, 10007]]

from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
my_pred = sc.fit_transform(my_prediction)

pred = model.predict(my_pred)
print(pred)

if pred == 0:
    print("Exonerated")
else:
    print("Unsubstantiated")

[1]
Exonerated
